<a href="https://colab.research.google.com/github/ilfpns/PhoSem/blob/main/Phosem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, glob, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch: 2.11.0+cu128
CUDA: True
Device: Tesla T4


In [ ]:
from google.colab import files

DATA_DIR = "/content/dataset"
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(DATA_DIR)
        print(f"압축 해제: {fname} → {DATA_DIR}")
    else:
        os.replace(fname, os.path.join(DATA_DIR, fname))
        print(f"이동: {fname} → {DATA_DIR}")

KeyboardInterrupt: 

In [ ]:
import re

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")

def parse_filename(path):
    """
    '429-x_die800.png' → {'num': 429, 'label': 'bad', 'is_die': True}
    '431-o.png'        → {'num': 431, 'label': 'good', 'is_die': False}
    규칙에 안 맞으면 None
    """
    name = os.path.splitext(os.path.basename(path))[0].lower()
    m = re.match(r"^(\d+)[-_]([ox])(.*)$", name)
    if not m:
        return None
    num, ox, rest = m.groups()
    return {
        "path": path,
        "num": int(num),
        "label": "good" if ox == "o" else "bad",
        "is_die": "die" in rest,
    }

all_files = sorted(
    p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
    if p.lower().endswith(IMG_EXTS)
)

parsed    = [info for p in all_files if (info := parse_filename(p))]
unparsed  = [p for p in all_files if parse_filename(p) is None]

die_files  = [f for f in parsed if f["is_die"]]
good_paths = sorted(f["path"] for f in die_files if f["label"] == "good")
bad_paths  = sorted(f["path"] for f in die_files if f["label"] == "bad")

print(f"전체 이미지        : {len(all_files)}장")
print(f"  ├ die 포함(사용) : {len(die_files)}장  →  양품 {len(good_paths)} / 불량 {len(bad_paths)}")
print(f"  ├ die 미포함(제외): {len(parsed) - len(die_files)}장")
print(f"  └ 규칙 불일치     : {len(unparsed)}장")

if unparsed:
    print("\n[확인 필요] 파일명 규칙에 안 맞는 파일:")
    for p in unparsed[:10]:
        print("   ", os.path.basename(p))

assert good_paths, "양품(die) 이미지가 없습니다 — PatchCore 메모리 뱅크에 필수"
print("\n양품 예시:", [os.path.basename(p) for p in good_paths[:3]])
print("불량 예시:", [os.path.basename(p) for p in bad_paths[:3]])

전체 이미지        : 0장
  ├ die 포함(사용) : 0장  →  양품 0 / 불량 0
  ├ die 미포함(제외): 0장
  └ 규칙 불일치     : 0장


AssertionError: 양품(die) 이미지가 없습니다 — PatchCore 메모리 뱅크에 필수

In [ ]:
def show_row(paths, title, n=4):
    n = min(n, len(paths))
    if n == 0:
        print(f"{title}: 없음"); return
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    axes = [axes] if n == 1 else axes
    for ax, p in zip(axes, paths[:n]):
        t, _ = load_image_gpu(p)
        ax.imshow(t[0].permute(1, 2, 0).cpu().numpy())
        ax.set_title(os.path.basename(p), fontsize=9); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_row(good_paths, "양품 (o, die)")
show_row(bad_paths,  "불량 (x, die)")